# 00 · Setup and verify

Welcome! This workshop series teaches **Flyte v2 / Union 2.x** authoring patterns. This first notebook makes sure your laptop can
talk to the control plane and that a remote run works end-to-end.

**Learning goals**

1. Configure the `flyte` CLI/SDK against your control plane
2. Run a first task remotely, straight from a notebook cell
3. Understand *how* notebook code executes remotely (and the four authoring rules that follow)
4. Verify the Image Builder can push to your Artifact Registry repo

**Prerequisites** (see [README](./README.md) and [appendix A](./appendix/A-platform-config-map.md)):
Python **3.12** kernel · control plane endpoint + credentials from your platform team ·
a project/domain created for the workshop · an Artifact Registry repo the dataplane can push/pull.

## 1. Install driver-side dependencies

Everything your *laptop* needs is pinned in `requirements.txt`. Task images declare their own
dependencies separately (you'll see how in section 5).

In [3]:
# Run once per environment (or: pip install -r requirements.txt in your venv)
! uv pip install -q -r requirements.txt

In [4]:
! flyte --version

Flyte SDK version: 2.5.7


## 2. Connect to your control plane

Copy [`config-templates/config.yaml.example`](./config-templates/config.yaml.example) to
`~/.flyte/config.yaml` and set your endpoint, or generate one:

```bash
flyte create config \
    --endpoint my-org.my-company.com \
    --domain development \
    --project flytesnacks \
    --builder remote
```

Every notebook in this series then connects with a single line: `flyte.init_from_config()`.
The first call opens a browser window for SSO (PKCE against your IdP).

In [ ]:
import flyte

flyte.init_from_config()

# Alternative, explicit form (useful when testing a second cluster or in CI):
# flyte.init(
#     endpoint="dns:///union.example.internal",
#     org="your-org",
#     project="onboarding",
#     domain="development",
# )

In [8]:
# Client-side workshop settings (.env) — project, domain, registry, buckets.
# NOTE: this module is CLIENT-SIDE ONLY; never import it inside a task body.
from workshop_config import preflight

WS = preflight()

project=onboarding domain=development
registry=<unset> gcs_bucket=<unset>
NOTE: unset in .env: registry, gcs_bucket — some cells will need them.


In [ ]:
# Confirm the control plane answers and your project exists
!flyte get project

## 3. How notebook code runs remotely

When you call `flyte.run` from a notebook, the SDK detects the interactive session and ships a
**pickled code bundle** of your cell-defined tasks to the cluster — no `.py` file or git checkout
required. That convenience comes with four **authoring rules**:

1. **Define tasks and their helpers in notebook cells** (or installed packages). Functions
   imported from local modules (like `workshop_config.py`) are pickled *by reference* and will
   fail remotely with `ModuleNotFoundError`. Pass config values into tasks as inputs instead.
2. **The kernel's Python minor version must match the task image** — everything here pins 3.12.
3. **`flyte.deploy()` is not supported from interactive sessions.** Anything that needs a
   *deployment* (triggers/schedules, tasks referenced via `flyte.remote.Task.get`, connectors,
   apps) lives in [`scripts/`](./scripts/) and is driven with `!flyte deploy ...` shell cells.
4. **`Environment(include=...)` (bundling extra files) is not supported in interactive runs** —
   that's also a `scripts/` job.

Everything else — `TaskEnvironment`, fan-out, caching, reusable containers, Ray — works
exactly the same from a notebook as from a file.

## 4. Hello, remote world

A `TaskEnvironment` groups tasks that share an image, resources, and runtime settings.
The default image (`flyte.Image.from_debian_base()`) ships with the `flyte` SDK pre-installed,
so this first run needs no image build.

In [ ]:
env = flyte.TaskEnvironment(
    name="hello",
    resources=flyte.Resources(cpu="1", memory="512Mi"),
)


@env.task
async def greet(name: str) -> str:
    return f"Hello from your self-managed dataplane, {name}!"

In [ ]:
run = await flyte.run.aio(greet, name="GCP")
print(run.name)
print(run.url)   # open this in the Union UI

In [ ]:
await run.wait.aio()          # block until the run finishes
run.outputs()

### At the Kubernetes level

That run created a pod in the `<project>-<domain>` namespace of your GKE dataplane cluster:

```bash
kubectl get pods -n onboarding-development
```

The pod ran as the `union` KubernetesServiceAccount, which your platform team bound to a Google
Service Account via Workload Identity — that's what lets tasks read/write GCS without keys.

## 5. Verify the Image Builder → Artifact Registry path

Real workloads need dependencies, so images get built. On a self-managed deployment the
**remote Image Builder** builds in-cluster and pushes to *your* Artifact Registry repo
(the `GAR_REGISTRY` value in `.env`). This cell verifies that whole path: build → push → pull → run.

The first execution takes a few minutes (image build); re-running is instant because image
builds are content-addressed and cached.

In [ ]:
image = (
    flyte.Image.from_debian_base(name="workshop-smoke", python_version=(3, 12))
    .with_pip_packages("pyjokes==0.8.3")
)

smoke_env = flyte.TaskEnvironment(
    name="image_smoke",
    image=image,
    resources=flyte.Resources(cpu="1", memory="512Mi"),
)


@smoke_env.task
async def tell_joke() -> str:
    import pyjokes
    return pyjokes.get_joke()


run = await flyte.run.aio(tell_joke)
print(run.url)
await run.wait.aio()
print(run.outputs())

> **If the build fails** with a push/pull permission error, the dataplane's builder or the task
> KSA is missing `roles/artifactregistry.writer`/`reader` on the repo — see
> [appendix A](./appendix/A-platform-config-map.md) → *Image Builder*.

## Further reading

- [Notebooks in the Union docs](https://www.union.ai/docs/v2/) → User guide → Task programming → Notebooks
- Next: [01-authoring-fundamentals](./01-authoring-fundamentals.ipynb)